In [1]:
# TODO(gbm): To to converted into a tutorial when the full path is ready.

In [2]:
import dgf
import tensorflow as tf

In [3]:
# Load a dataset
graph, schema = dgf.io.fetch_ogb_graph("arxiv")

Caching arxiv graph at /tmp/gf_fetch/arxiv.cache


In [4]:
# Train a model
model = dgf.learning.train_node_model(
    graph=graph,
    schema=schema,
    target_column="labels",
    verbose=1,
    max_training_time_seconds=10,
)

Preparing dataset
Num. training seed nodes: 152409, Num. validation seed nodes: 16934


[Warning] No normalizer created for node set 'nodes', feature '#id'.


Preparing dataset finished in 5.19 seconds
Caching validation dataset
Caching validation dataset finished in 4.61 seconds
Number of cache validation batches: 528
Training model
Generate first batch to initialize model
Create model variables
...Tracing model
Will validate model every 1000 step(s)
Will checkpoint model every 1000 step(s)
Start training


Training:   0%|          | 0/10000 [00:00<?, ?it/s]

...Tracing model


...

Training:   0%|          | 29/10000 [00:10<36:00,  4.62it/s, step=0, train-accuracy=0.0000, train-loss=8.5882]

Max training time of 10 seconds exceeded. Stopping training.


Training:   0%|          | 29/10000 [00:10<58:27,  2.84it/s, step=0, train-accuracy=0.0000, train-loss=8.5882]

...Tracing model


Validation loop took 17.81s (only printed once)
Final metrics: {'step': '0', 'train-accuracy': '0.0000', 'train-loss': '8.5882'}
Training model finished in 34.77 seconds


In [5]:
# Extract and instantiate a sampler from the model
# Note: This is currently not part of the API. This will be at some point.
# TODO(gbm): Normalize the way to access the sampling_plan and schema as part of
# the API.

print("The sampling plan:")
print(model.data().sampling_plan)

sampler = dgf.sampling.create_sampler(
    graph=graph,
    plan=model.data().sampling_plan,
    schema=model.data().schema,
    batch_size=2,
)

The sampling plan:
SamplingPlan(root='nodes')
Plan:
  - edges [width=15] -> nodes
    - edges [width=15] -> nodes
    - edges (reversed) [width=15] -> nodes
  - edges (reversed) [width=15] -> nodes
    - edges [width=15] -> nodes
    - edges (reversed) [width=15] -> nodes


In [6]:
# Generate a graph sample for the first node
sample = sampler.sample(0)

In [7]:
# Make a prediction on this sample (just to check)
model.predict_on_graph_sample_batch([sample])

...Tracing model


array([[3.04979017e-06, 2.88427179e-03, 3.40165608e-02, 3.79447057e-03,
        9.58636883e-06, 7.13639287e-03, 2.75157345e-03, 3.88313900e-04,
        1.56331360e-01, 7.30279789e-05, 1.08684385e-02, 1.58047842e-05,
        1.98892826e-06, 4.67464160e-05, 1.12107176e-04, 1.87232457e-02,
        2.65875220e-01, 2.64645251e-03, 1.21548194e-02, 8.31147631e-08,
        8.05679301e-05, 4.94736014e-07, 5.41739026e-03, 2.22300482e-03,
        1.26531096e-02, 6.99983575e-05, 3.16237584e-02, 2.56000523e-04,
        2.76719294e-02, 2.58401632e-01, 2.26971795e-04, 6.02786057e-03,
        7.01011764e-03, 1.36003131e-02, 8.13177787e-03, 1.86563339e-04,
        4.99112730e-06, 1.08529329e-01, 3.43710562e-05, 1.63165605e-05]],
      dtype=float32)

In [8]:
# Export to TF and generate a prediction
# Note: The result does not need any custom ops.
tf_model = model.to_tensorflow_function()

In [9]:
# Convert the sample into a tf sample
# Note: This method is simply converting the numpy arrays into tensorflow arrays
# i.e. it is relatively efficient.
tf_sample = dgf.convert.graph_to_tf_graph(sample, model.data().schema)

In [10]:
# Make a prediction with the tf model
tf_model(tf_sample, tf.constant([0]))

...Tracing model


<tf.Tensor: shape=(1, 40), dtype=float32, numpy=
array([[3.04980290e-06, 2.88427551e-03, 3.40165086e-02, 3.79446475e-03,
        9.58638975e-06, 7.13642780e-03, 2.75157439e-03, 3.88314395e-04,
        1.56331554e-01, 7.30281390e-05, 1.08684776e-02, 1.58047733e-05,
        1.98892894e-06, 4.67465179e-05, 1.12107213e-04, 1.87232699e-02,
        2.65875280e-01, 2.64645321e-03, 1.21548111e-02, 8.31151041e-08,
        8.05679520e-05, 4.94737549e-07, 5.41739212e-03, 2.22300761e-03,
        1.26531255e-02, 6.99983793e-05, 3.16237807e-02, 2.56001309e-04,
        2.76719648e-02, 2.58400947e-01, 2.26972086e-04, 6.02787267e-03,
        7.01012462e-03, 1.36003112e-02, 8.13180767e-03, 1.86563571e-04,
        4.99114367e-06, 1.08529575e-01, 3.43711326e-05, 1.63165969e-05]],
      dtype=float32)>

In [11]:
# Save the model
tf.saved_model.save(tf_model, "/tmp/model")

...Tracing model
...Tracing model


In [12]:
# Load the model
loaded_model = tf.saved_model.load("/tmp/model")


def show_signature(module):
  for signature_key in module.signatures:
    print(f"\nSignature Key: '{signature_key}'")

    signature = module.signatures[signature_key]
    print("  Inputs:")
    for input_tensor in signature.inputs:
      print(
          f"    - {input_tensor.name}, Shape: {input_tensor.shape}, DType:"
          f" {input_tensor.dtype}"
      )
    print("  Outputs:")
    for output_key, output_tensor in signature.structured_outputs.items():
      print(
          f"    - '{output_key}', Name: {output_tensor.name}, Shape:"
          f" {output_tensor.shape}, DType: {output_tensor.dtype}"
      )


show_signature(loaded_model)


Signature Key: 'serving_default'
  Inputs:
    - graph:0, Shape: (), DType: <dtype: 'int32'>
    - graph_1:0, Shape: (None,), DType: <dtype: 'string'>
    - graph_2:0, Shape: (None,), DType: <dtype: 'string'>
    - graph_3:0, Shape: (None, 128), DType: <dtype: 'float32'>
    - graph_4:0, Shape: (None,), DType: <dtype: 'int64'>
    - graph_5:0, Shape: (None,), DType: <dtype: 'int64'>
    - graph_6:0, Shape: (2, None), DType: <dtype: 'int64'>
    - seed_node_idxs:0, Shape: (None,), DType: <dtype: 'int32'>
    - unknown:0, Shape: (), DType: <dtype: 'resource'>
    - unknown_0:0, Shape: (), DType: <dtype: 'int64'>
  Outputs:
    - 'output_0', Name: output_0, Shape: (None, 40), DType: <dtype: 'float32'>


In [13]:
# Re-export the model as tf function, but this time, the function consumes a flat dict.
tf_model_flat = model.to_tensorflow_function(consume_tf_graph_dict=True)

...Tracing model


In [14]:
# Convert our tf graph into a tf graph dict (a flat dict).
tf_flat_sample = dgf.convert.tf_graph_to_tf_graph_dict(tf_sample)

In [15]:
tf_flat_sample.keys()

dict_keys(['nodes__nodes__reserved_size', 'nodes__nodes___hash_id', 'nodes__nodes___hash_split', 'nodes__nodes__labels', 'nodes__nodes__year', 'nodes__nodes__feat', 'edges__edges__reserved_adjacency'])

In [16]:
# Save and restore the model
tf.saved_model.save(tf_model_flat, "/tmp/model_flat2")
loaded_tf_model_flat = tf.saved_model.load("/tmp/model_flat2")

...Tracing model


In [17]:
loaded_tf_model_flat.signatures

_SignatureMap({'serving_default': <ConcreteFunction (*, edges__edges__reserved_adjacency: TensorSpec(shape=(2, None), dtype=tf.int64, name='edges__edges__reserved_adjacency'), nodes__nodes___hash_id: TensorSpec(shape=(None,), dtype=tf.string, name='nodes__nodes___hash_id'), nodes__nodes___hash_split: TensorSpec(shape=(None,), dtype=tf.string, name='nodes__nodes___hash_split'), nodes__nodes__feat: TensorSpec(shape=(None, 128), dtype=tf.float32, name='nodes__nodes__feat'), nodes__nodes__labels: TensorSpec(shape=(None,), dtype=tf.int64, name='nodes__nodes__labels'), nodes__nodes__reserved_size: TensorSpec(shape=(), dtype=tf.int32, name='nodes__nodes__reserved_size'), nodes__nodes__year: TensorSpec(shape=(None,), dtype=tf.int64, name='nodes__nodes__year'), seed_node_idxs: TensorSpec(shape=(None,), dtype=tf.int32, name='seed_node_idxs')) -> Dict[['output_0', TensorSpec(shape=(None, 40), dtype=tf.float32, name='output_0')]] at 0x7386BA45B850>})

In [18]:
loaded_tf_model_flat.signatures["serving_default"]

<ConcreteFunction (*, edges__edges__reserved_adjacency: TensorSpec(shape=(2, None), dtype=tf.int64, name='edges__edges__reserved_adjacency'), nodes__nodes___hash_id: TensorSpec(shape=(None,), dtype=tf.string, name='nodes__nodes___hash_id'), nodes__nodes___hash_split: TensorSpec(shape=(None,), dtype=tf.string, name='nodes__nodes___hash_split'), nodes__nodes__feat: TensorSpec(shape=(None, 128), dtype=tf.float32, name='nodes__nodes__feat'), nodes__nodes__labels: TensorSpec(shape=(None,), dtype=tf.int64, name='nodes__nodes__labels'), nodes__nodes__reserved_size: TensorSpec(shape=(), dtype=tf.int32, name='nodes__nodes__reserved_size'), nodes__nodes__year: TensorSpec(shape=(None,), dtype=tf.int64, name='nodes__nodes__year'), seed_node_idxs: TensorSpec(shape=(None,), dtype=tf.int32, name='seed_node_idxs')) -> Dict[['output_0', TensorSpec(shape=(None, 40), dtype=tf.float32, name='output_0')]] at 0x7386BA45B850>

In [19]:
# Check the signature. It should be flat
show_signature(loaded_tf_model_flat)


Signature Key: 'serving_default'
  Inputs:
    - edges__edges__reserved_adjacency:0, Shape: (2, None), DType: <dtype: 'int64'>
    - nodes__nodes___hash_id:0, Shape: (None,), DType: <dtype: 'string'>
    - nodes__nodes___hash_split:0, Shape: (None,), DType: <dtype: 'string'>
    - nodes__nodes__feat:0, Shape: (None, 128), DType: <dtype: 'float32'>
    - nodes__nodes__labels:0, Shape: (None,), DType: <dtype: 'int64'>
    - nodes__nodes__reserved_size:0, Shape: (), DType: <dtype: 'int32'>
    - nodes__nodes__year:0, Shape: (None,), DType: <dtype: 'int64'>
    - seed_node_idxs:0, Shape: (None,), DType: <dtype: 'int32'>
    - unknown:0, Shape: (), DType: <dtype: 'resource'>
    - unknown_0:0, Shape: (), DType: <dtype: 'int64'>
  Outputs:
    - 'output_0', Name: output_0, Shape: (None, 40), DType: <dtype: 'float32'>


In [20]:
# Generate a prediction
loaded_tf_model_flat(**tf_flat_sample, seed_node_idxs=tf.constant([0]))

<tf.Tensor: shape=(1, 40), dtype=float32, numpy=
array([[3.04980290e-06, 2.88427551e-03, 3.40165086e-02, 3.79446475e-03,
        9.58638975e-06, 7.13642780e-03, 2.75157439e-03, 3.88314395e-04,
        1.56331554e-01, 7.30281390e-05, 1.08684776e-02, 1.58047733e-05,
        1.98892894e-06, 4.67465179e-05, 1.12107213e-04, 1.87232699e-02,
        2.65875280e-01, 2.64645321e-03, 1.21548111e-02, 8.31151041e-08,
        8.05679520e-05, 4.94737549e-07, 5.41739212e-03, 2.22300761e-03,
        1.26531255e-02, 6.99983793e-05, 3.16237807e-02, 2.56001309e-04,
        2.76719648e-02, 2.58400947e-01, 2.26972086e-04, 6.02787267e-03,
        7.01012462e-03, 1.36003112e-02, 8.13180767e-03, 1.86563571e-04,
        4.99114367e-06, 1.08529575e-01, 3.43711326e-05, 1.63165969e-05]],
      dtype=float32)>